# Multiclass Logistic Regression & Coefficient Interpretation Lab

Binary logistic regression classifies two categories ($y \in \{0, 1\}$). For multiclass tasks ($K > 2$), we employ either heuristic ensembles like **One-vs-Rest (OvR)** or unified probabilistic models like **Multinomial Softmax**. This lab benchmarks OvR against Softmax on the Iris dataset, examines probability normalization, plots decision regions, and computes odds ratios from model coefficients.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

## 1. Probability Normalization: OvR vs. Softmax

Fit both `multi_class='ovr'` and `multi_class='multinomial'` on a 12-sample synthetic dataset to examine whether output class probabilities sum to 1.0.

In [ ]:
X_toy = np.array([
    [2, 1], [2, 2], [3, 1], [2, 3],  # Class 0
    [5, 5], [6, 4], [4, 5], [6, 5],  # Class 1
    [9, 9], [9, 10], [8, 9], [9, 8]  # Class 2
])
y_toy = np.array([0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2])

clf_ovr = LogisticRegression(multi_class='ovr', random_state=42, max_iter=1000).fit(X_toy, y_toy)
clf_soft = LogisticRegression(multi_class='multinomial', random_state=42, max_iter=1000).fit(X_toy, y_toy)

p_ovr = clf_ovr.predict_proba(X_toy[:4])
p_soft = clf_soft.predict_proba(X_toy[:4])

print(f"{'Sample':<8} {'OvR Probabilities (P0, P1, P2)':<35} {'OvR Sum':<10} {'Softmax Sum':<12}")
print("-" * 68)
for i in range(4):
    ovr_str = f"[{p_ovr[i,0]:.3f}, {p_ovr[i,1]:.3f}, {p_ovr[i,2]:.3f}]"
    print(f"{i:<8} {ovr_str:<35} {p_ovr[i].sum():<10.4f} {p_soft[i].sum():<12.4f}")

## 2. Real-World Multiclass Benchmark on Iris

Train both classifiers on the full Iris dataset (150 samples, 3 classes) and compare accuracy and confusion matrices.

In [ ]:
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

m_ovr = LogisticRegression(multi_class='ovr', max_iter=1000, random_state=42).fit(X_iris, y_iris)
m_soft = LogisticRegression(multi_class='multinomial', max_iter=1000, random_state=42).fit(X_iris, y_iris)

acc_ovr = accuracy_score(y_iris, m_ovr.predict(X_iris))
acc_soft = accuracy_score(y_iris, m_soft.predict(X_iris))

print(f"One-vs-Rest Accuracy:    {acc_ovr:.2%}")
print(f"Multinomial Accuracy:    {acc_soft:.2%}")

print("\nConfusion Matrix (Softmax):")
print(confusion_matrix(y_iris, m_soft.predict(X_iris)))

## 3. Interpreting Coefficients as Odds Ratios

Fit a binary model on Class 0 vs Rest to extract feature weights $w$ and compute multiplicative Odds Ratios ($e^w$).

In [ ]:
y_bin = (y_iris == 0).astype(int)
clf_bin = LogisticRegression(random_state=42).fit(X_iris, y_bin)

print(f"{'Feature Name':<20} {'Weight (w)':<14} {'Odds Ratio (e^w)':<18} {'Direction'}")
print("-" * 68)
for name, w in zip(iris.feature_names, clf_bin.coef_[0]):
    or_val = np.exp(w)
    direction = "Increases Odds" if w > 0 else "Decreases Odds"
    print(f"{name:<20} {w:<14.4f} {or_val:<18.4f} {direction}")